# Cold-start extend on LeiData wav1 · window 1

Views the last cold-start `extend_dictionary` experiment:

- **WAV:** LeiData wav1 (`SOCAL_H_76_230427_003345.x.wav`)
- **Windows:** `--n-windows 1` (offset 0)
- **Knobs:** `--min-repeats 2`, `--max-repeats 5`, `--eps-step-fraction 0.5`, `--max-new-atoms 10`
- **Summary JSON:** `stage_w1_mr2_max5.json` under the cold-start out dir

Earlier cold starts in the same folder (`stage_w1.json`, `stage_w1_mr2.json`) are listed for comparison.

In [ ]:
%matplotlib inline

import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

REPO = Path("/home/yfreund/projects/gplrev3-webserver")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

LIBRARY_ROOT = Path("/home/yfreund/whale_processing")
OUT_ROOT = LIBRARY_ROOT / "result/LeiData/dict_learning_templates/wav1_extend_w1_k1_cold"
DB_PATH = LIBRARY_ROOT / "result/dict_learning_catalog.sqlite"
WAV = LIBRARY_ROOT / "LeiData/SOCAL_H_76_230427_003345.x.wav"

# Last experiment (min_repeats=2, max_repeats=5)
STAGE_JSON = OUT_ROOT / "stage_w1_mr2_max5.json"


def load_stage(path: Path) -> dict:
    summary = json.loads(path.read_text())
    out_dir = Path(summary["out_dir"])
    man_path = out_dir / "manifest.json"
    man = json.loads(man_path.read_text()) if man_path.is_file() else {}
    params = summary.get("params") or man.get("params") or {}
    e0 = man.get("energy_before")
    e1 = man.get("energy_after")
    frac = None
    if e0 is not None and e1 is not None and float(e0) > 0:
        frac = (float(e0) - float(e1)) / float(e0)
    return {
        "label": path.stem,
        "path": path,
        "summary": summary,
        "manifest": man,
        "out_dir": out_dir,
        "params": params,
        "coverage": float(summary.get("coverage") or 0.0),
        "n_atoms": int(summary.get("n_atoms") or len(man.get("templates") or [])),
        "n_atoms_added": int(summary.get("n_atoms_added") or 0),
        "stop_reason": summary.get("stop_reason"),
        "dictionary_id": summary.get("dictionary_id"),
        "energy_before": e0,
        "energy_after": e1,
        "fraction_removed": frac,
        "n_matchings": len(man.get("matchings") or []),
        "min_repeats": params.get("min_repeats"),
        "max_repeats": params.get("max_repeats"),
    }


stage = load_stage(STAGE_JSON)
print(f"Stage JSON: {STAGE_JSON}")
print(f"out_dir:    {stage['out_dir']}")
print(f"dictionary_id={stage['dictionary_id']}  atoms={stage['n_atoms']}  "
      f"matchings={stage['n_matchings']}  coverage={stage['coverage']:.3f}  "
      f"stop={stage['stop_reason']}")

## Run summary

In [ ]:
p = stage["params"]
pct = (
    "—"
    if stage["fraction_removed"] is None
    else f"{100 * stage['fraction_removed']:.1f}%"
)
lines = [
    "| Field | Value |",
    "|---|---|",
    f"| `dictionary_id` | **{stage['dictionary_id']}** |",
    f"| `source` | `{stage['summary'].get('source')}` |",
    f"| `n_windows` | {p.get('n_windows')} |",
    f"| `min_repeats` / `max_repeats` | {p.get('min_repeats')} / {p.get('max_repeats')} |",
    f"| `eps_step_fraction` | {p.get('eps_step_fraction')} |",
    f"| `max_new_atoms` | {p.get('max_new_atoms')} |",
    f"| `min_coverage` | {p.get('min_coverage')} |",
    f"| `n_atoms` / added | {stage['n_atoms']} / {stage['n_atoms_added']} |",
    f"| `n_matchings` | {stage['n_matchings']} |",
    f"| coverage | {stage['coverage']:.4f} |",
    f"| %E removed | {pct} |",
    f"| `stop_reason` | `{stage['stop_reason']}` |",
    f"| `out_dir` | `{stage['out_dir']}` |",
]
display(Markdown("\n".join(lines)))

logs = stage["summary"].get("stage_logs") or []
if logs:
    print("stage_logs:")
    for row in logs:
        print(" ", row)

## Compare cold-start variants in this folder

In [ ]:
variants = []
for path in sorted(OUT_ROOT.glob("stage_w1*.json")):
    try:
        variants.append(load_stage(path))
    except (json.JSONDecodeError, KeyError, OSError) as exc:
        print(f"skip {path.name}: {exc}")

lines = [
    "| file | min_r | max_r | atoms | matchings | coverage | %E | stop | dict_id |",
    "|---|---:|---:|---:|---:|---:|---:|---|---|",
]
for v in variants:
    pct = "—" if v["fraction_removed"] is None else f"{100 * v['fraction_removed']:.1f}%"
    highlight = "**" if v["path"] == STAGE_JSON else ""
    lines.append(
        f"| {highlight}`{v['label']}`{highlight} | {v['min_repeats']} | {v['max_repeats']} | "
        f"{v['n_atoms']} | {v['n_matchings']} | {v['coverage']:.3f} | {pct} | "
        f"`{v['stop_reason']}` | {v['dictionary_id']} |"
    )
display(Markdown("\n".join(lines)))

## Learned template(s)

In [ ]:
man = stage["manifest"]
out_dir = stage["out_dir"]
templates = []
for item in man.get("templates") or []:
    arr = np.load(out_dir / item["file"])
    view = arr[0] if arr.ndim == 3 else arr
    templates.append((int(item["template_id"]), view, arr.ndim, item))

n = len(templates)
if n == 0:
    print("No templates in manifest.")
else:
    ncols = min(4, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.0 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for i, (tid, view, ndim, meta) in enumerate(templates):
        ax = axes[i // ncols][i % ncols]
        ax.imshow(view, aspect="auto", origin="lower", cmap="magma")
        w = meta.get("width") or view.shape[-1]
        ax.set_title(f"id={tid}  {view.shape[0]}×{w}" + (" (mode0)" if ndim == 3 else ""), fontsize=10)
        ax.set_xlabel("time"); ax.set_ylabel("freq")
        ax.axis("on")
    fig.suptitle(
        f"Templates · dict {stage['dictionary_id']} · "
        f"min_repeats={stage['min_repeats']} max_repeats={stage['max_repeats']}",
        y=1.02,
    )
    fig.tight_layout()
    plt.show()

counts = Counter(int(m["template_id"]) for m in man.get("matchings") or [])
if counts:
    tids = sorted(counts)
    fig, ax = plt.subplots(figsize=(max(3.5, 0.5 * len(tids)), 3))
    ax.bar([str(t) for t in tids], [counts[t] for t in tids], color="steelblue")
    ax.set_xlabel("template_id"); ax.set_ylabel("# matchings")
    ax.set_title("Matchings per atom")
    for tid in tids:
        ax.text(str(tid), counts[tid], str(counts[tid]), ha="center", va="bottom")
    fig.tight_layout()
    plt.show()

## Window-1 spectrogram with matching overlays

Load the first whitened window from the catalog and draw each matching as a colored time span.

In [ ]:
from database import catalog


def first_window_array(db_path: Path, wav: Path) -> tuple[np.ndarray, str]:
    with catalog.connect(db_path) as conn:
        wav_row = catalog.find_wav_file(conn, str(wav))
        if wav_row is None:
            raise FileNotFoundError(wav)
        segs = catalog.list_gpl_segmentations_for_wav(conn, wav_row["wav_id"])
        wins = catalog.list_windows_for_segmentation(
            conn, segs[0]["segmentation_id"], limit=1, offset=0
        )
        npy = Path(wins[0]["npy_path"])
        return np.load(npy), wins[0]["window_id"]


def template_widths(man: dict) -> dict[int, int]:
    widths = {}
    for t in man.get("templates") or []:
        tid = int(t["template_id"])
        if t.get("width") is not None:
            widths[tid] = int(t["width"])
        else:
            shape = t.get("shape") or []
            widths[tid] = int(shape[-1]) if shape else 1
    return widths


spec, wid = first_window_array(DB_PATH, WAV)
print(f"Window: {wid}  shape={spec.shape}")

widths = template_widths(man)
matchings = [m for m in (man.get("matchings") or []) if m.get("window_id") == wid]
cmap = plt.cm.tab10

fig, axes = plt.subplots(2, 1, figsize=(14, 6.5), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]})
ax0, ax1 = axes
ax0.imshow(spec, aspect="auto", origin="lower", cmap="gray")
for m in matchings:
    tid = int(m["template_id"])
    t0 = int(m["t_start"])
    w = widths.get(tid, 1)
    ax0.axvspan(t0, t0 + w, color=cmap(tid % 10), alpha=0.35)
    ax0.axvline(t0, color=cmap(tid % 10), lw=0.8, alpha=0.8)
ax0.set_ylabel("freq bin")
ax0.set_title(
    f"dict {stage['dictionary_id']} · {len(matchings)} matchings on window 1 · "
    f"coverage={stage['coverage']:.3f} · stop=`{stage['stop_reason']}`"
)

# Timeline of placements
ax1.set_xlim(0, spec.shape[1])
ax1.set_ylim(-0.5, max(len(widths), 1) - 0.5)
for m in matchings:
    tid = int(m["template_id"])
    t0 = int(m["t_start"])
    w = widths.get(tid, 1)
    ax1.barh(tid, w, left=t0, height=0.6, color=cmap(tid % 10), alpha=0.85)
ax1.set_xlabel("time bin")
ax1.set_ylabel("template_id")
ax1.set_yticks(sorted(widths) if widths else [0])
ax1.set_title("Matching intervals")
fig.tight_layout()
plt.show()

print("Matchings:")
for m in sorted(matchings, key=lambda x: (x["t_start"], x["template_id"])):
    tid = int(m["template_id"])
    t0 = int(m["t_start"])
    w = widths.get(tid, 1)
    amp = m.get("amplitude")
    amp_s = f"{float(amp):.3f}" if amp is not None else "—"
    print(f"  tid={tid}  t=[{t0}, {t0 + w})  amp={amp_s}")

## Catalog cross-check

In [ ]:
dict_id = stage["dictionary_id"]
if dict_id is None:
    print("No catalog dictionary_id (subspace / --no-register-catalog?).")
else:
    with catalog.connect(DB_PATH) as conn:
        d = catalog.get_dictionary(conn, int(dict_id))
        tmpls = catalog.list_dictionary_templates(conn, int(dict_id))
        matches = catalog.list_matchings(conn, int(dict_id))
        wins = catalog.list_dictionary_windows(conn, int(dict_id))
    print(
        f"dictionary_id={dict_id}  name={d.get('name')!r}  source={d.get('source')}  "
        f"parent={d.get('parent_dictionary_id')}"
    )
    print(f"catalog: templates={len(tmpls)}  windows={len(wins)}  matchings={len(matches)}")
    assert len(tmpls) == stage["n_atoms"], "template count mismatch vs stage JSON"
    assert len(matches) == stage["n_matchings"], "matching count mismatch vs manifest"
    print("OK — catalog agrees with stage/manifest counts.")